# Search and Sort Workloads


```{contents}
:local:
:depth: 2
```


An algorithm choice should answer a business requirement, not just win one small timing demonstration. A daily report, a one-time lookup, and an interactive index have different workloads even when all use the same records.

Combine searching, ordering, copying, and validation into one task. First say what may change and what the caller must receive. Then choose an operation and explain both its preparation cost and its repeated-use cost.


## Preserve Arrival Order While Preparing a Report

```{index} Preserve Arrival Order While Preparing a Report
```


Sort a copy when the original sequence records arrival order. An array clone is shallow: it copies element values or object references, not every reachable object. Sorting the new array changes its order without changing the original array's order.

This distinction extends Chapter 12's ownership discussion. A report snapshot is not an independent copy of mutable invoice objects. Decide whether records themselves may change while the report is used.


In [1]:
using System;
using System.Collections.Generic;

int[] arrived = {300, 100, 200, 100};
int[] report = (int[])arrived.Clone();
BasicSort.InsertionSort(report, Comparer<int>.Default);
Console.WriteLine("Arrival: " + string.Join(",", arrived));
Console.WriteLine("Report: " + string.Join(",", report));
Console.WriteLine(BasicSearch.BinaryAny(report, 200, Comparer<int>.Default));

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


Arrival: 300,100,200,100
Report: 100,100,200,300
2


## Include Preparation and Updates

```{index} Include Preparation and Updates
```


For q searches over n records, repeated linear scans can require qn equality checks. A prepared sorted array needs sorting work plus q logarithmic searches. In this chapter's insertion-sort implementation, worst-case preparation is quadratic; a standard efficient comparison sort has a different preparation bound. Do not assign a library sort's cost to the teaching insertion sort.

The benefit also depends on updates. A static end-of-day report can be prepared once. An interactive record set may require maintaining order or rebuilding a snapshot. Later hash collections provide another option for exact-key lookup; trees support ordered access. There is no single structure that is best for every operation.


## Library Operations and Clear Comparers

```{index} Library Operations and Clear Comparers
```


Array.Sort and List.Sort are ordinary application tools. Supply the ordering policy when sorting custom types, and preserve input if mutation is forbidden. Comparers should be consistent: a value compares equal to itself, reversing operands reverses the ordering sign, and the ordering should not form contradictory cycles.

For displayed text, choose an ordering suitable for users. For symbolic keys in the core examples, ordinal comparison makes the rule explicit. A search must use the same ordering that created its sorted input.


In [1]:
using System;
using System.Collections.Generic;

string[] ids = {"T-200", "T-010", "T-100"};
Array.Sort(ids, StringComparer.Ordinal);
Console.WriteLine(string.Join(",", ids));
Console.WriteLine(Array.BinarySearch(ids, "T-100", StringComparer.Ordinal));


T-010,T-100,T-200
1


## Design a Boundary Matrix

```{index} Design a Boundary Matrix
```


A small check matrix can include empty input, one element, a missing key before or after the range, duplicates, all-equal values, sorted input, and reverse input. For object sorting, check a stable-tie example against the required policy.

Keep each result contract explicit. A first-match linear search, any-match binary search, and “all matching invoices” query are different tasks. Do not claim one implementation is faster by giving it less work or a weaker required answer.

In the connected lab, prepare a ticket report, sort it, search it, and check preserved input and duplicate counts. In homework, transfer those skills to rosters, amounts, and differently ordered records. The next chapters use ordered operations inside trees and priority structures.


## Practice: Choose a Safe Reporting Snapshot

```{index} Practice: Choose a Safe Reporting Snapshot
```


Create an independently ordered array before sorting; the original arrival order must remain unchanged.


In [ ]:
using System;
using System.Collections.Generic;

int[] original = {40, 10, 40};
int[] sorted = original /* TODO: copy the outer array */;
BasicSort.InsertionSort(sorted, Comparer<int>.Default);
if (original[0] != 40 || sorted.Length != original.Length) throw new Exception("Snapshot failed.");
Console.WriteLine(string.Join(",", sorted));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


In [1]:
using System;
using System.Collections.Generic;

int[] original = {40, 10, 40};
int[] sorted = (int[])original.Clone();
BasicSort.InsertionSort(sorted, Comparer<int>.Default);
if (original[0] != 40 || sorted.Length != original.Length) throw new Exception("Snapshot failed.");
Console.WriteLine(string.Join(",", sorted));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


10,40,40


```{rubric} Footnotes
```
[^1]: Measured timings depend on the runtime and inputs. Operation counts and stated assumptions support a growth argument; one fast run does not establish a general complexity bound.
